In [1]:
import requests
from PIL import Image
import gradio as gr
import io

# Custom Vision Prediction 키와 URL

# Gradio 인터페이스 정의


/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import requests                  # HTTP 요청을 보내기 위한 라이브러리
from PIL import Image            # 이미지 처리용 라이브러리 (Pillow)
import gradio as gr              # Gradio UI 생성 라이브러리
import io                        # 바이트 스트림 처리를 위한 모듈

# Custom Vision Prediction 서비스 키와 엔드포인트 URL 설정
PREDICTION_KEY = ""
ENDPOINT_URL   = ""
# API 호출에 사용할 헤더, Pr

# API 호출에 사용할 헤더, Prediction-Key와 데이터 타입 지정
headers = {
    "Prediction-Key": PREDICTION_KEY,
    "Content-Type": "application/octet-stream"   # 바이너리 이미지 데이터 전송
}

In [3]:
def predict_with_api(image: Image.Image):
    # PIL 이미지 객체를 JPEG 형식의 바이너리 데이터로 변환
    buf = io.BytesIO()                          # 메모리 내 바이트 버퍼 생성
    image.save(buf, format='JPEG')              # 이미지 데이터를 JPEG로 버퍼에 저장
    byte_data = buf.getvalue()                  # 버퍼에 저장된 바이너리 데이터 추출

    # Azure Custom Vision API에 POST 요청으로 이미지 전송
    response = requests.post(ENDPOINT_URL, headers=headers, data=byte_data)

    # JSON 형태로 응답 받아 예측 결과 파싱
    predictions = response.json()["predictions"]

    # 확률(probability)이 가장 높은 예측 항목을 선택
    top_prediction = max(predictions, key=lambda x: x["probability"])
    label = top_prediction["tagName"]                      # 예측된 클래스 이름
    probability = top_prediction["probability"]            # 예측 확률

    # 결과 문자열 포맷팅 후 반환 (예: 'cat (98.23%)')
    return f"{label} ({probability*100:.2f}%)"

In [4]:
# Gradio 인터페이스 구성
interface = gr.Interface(
    fn=predict_with_api,            # 이미지 입력을 받아 API 호출하는 함수 지정
    inputs=gr.Image(type="pil"),    # 입력 타입: PIL 이미지 객체
    outputs=gr.Text(),              # 출력 타입: 텍스트 (예측 결과)
    title="Custom Vision Image Classifier",   # 앱 제목
    description="Upload an image to see the prediction from your Custom Vision model." # 앱 설명
)

# 웹 서버를 실행하여 인터페이스 실행 (브라우저에서 접속 가능)
interface.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [5]:
import io
import requests
import gradio as gr
from PIL import Image


# ===== 예측 함수 =====
def predict_with_api(image: Image.Image):
    # 1. 이미지 미업로드 방어
    if image is None:
        return "이미지를 업로드해 주세요."

    # 2. PIL 이미지 → JPEG 바이트 변환
    buf = io.BytesIO()
    # JPEG는 RGB만 지원하므로 변환 (RGBA/팔레트 이미지 대비)
    if image.mode != "RGB":
        image = image.convert("RGB")
    image.save(buf, format="JPEG")
    byte_data = buf.getvalue()

    # 3. API 요청
    try:
        response = requests.post(ENDPOINT_URL, headers=headers, data=byte_data, timeout=15)
    except requests.exceptions.RequestException as e:
        return f"요청 실패: {e}"

    # 4. HTTP 상태 확인
    if response.status_code != 200:
        return f"API 에러 {response.status_code}: {response.text[:200]}"

    # 5. JSON 파싱
    try:
        result = response.json()
    except ValueError:
        return f"JSON 파싱 실패. 응답: {response.text[:200]}"

    # 6. 예측 결과 추출
    predictions = result.get("predictions", [])
    if not predictions:
        return "예측 결과가 없습니다."

    top = max(predictions, key=lambda x: x["probability"])
    label = top["tagName"]
    probability = top["probability"]

    return f"{label} ({probability * 100:.2f}%)"


# ===== Gradio 인터페이스 =====
interface = gr.Interface(
    fn=predict_with_api,
    inputs=gr.Image(type="pil"),
    outputs=gr.Text(),
    title="Custom Vision Image Classifier",
    description="Upload an image to see the prediction from your Custom Vision model.",
)

# 웹 서버 실행 (브라우저에서 접속 가능)
interface.launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
